In [10]:
# Mount google Drive

from google.colab import drive
drive.mount('/content/drive')


import os

print(os.path.exists("/content/drive/MyDrive"))
print(os.listdir("/content/drive") if os.path.exists("/content/drive") else "Not mounted")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
True
['.shortcut-targets-by-id', 'MyDrive', '.Trash-0', '.Encrypted']


In [11]:
# Install dependencies

!pip install -q \ sentence-transformers \ transformers \ accelerate \ pandas \ numpy \ tqdm \ faiss-cpu
!pip install -q faiss-cpu


#Imports

import os
import gc
import json
import pickle
import faiss
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer

import torch

In [12]:
#Configuration
PROJECT_ROOT = "/content/drive/MyDrive/RAGBenchmark"
DATASET_DIR = os.path.join(PROJECT_ROOT, "datasets")
CHUNK_DIR = os.path.join(PROJECT_ROOT, "chunks")
EMBEDDING_DIR = os.path.join(PROJECT_ROOT, "embeddings")
os.makedirs(EMBEDDING_DIR, exist_ok=True)
INDEX_DIR = os.path.join(PROJECT_ROOT, "indexes")
EXPERIMENT_DIR = os.path.join(PROJECT_ROOT, "experiments")
JUDGE_CACHE_DIR = os.path.join(PROJECT_ROOT, "judge_cache")
EMBEDDING_MODELS = { "llm_embedder": "BAAI/llm-embedder" }
BATCH_SIZE = 64
NORMALIZE = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Reading chunks from :", CHUNK_DIR)
print("Saving embeddings to:", EMBEDDING_DIR)


for folder in [
    DATASET_DIR,
    CHUNK_DIR,
    EMBEDDING_DIR,
    INDEX_DIR,
    EXPERIMENT_DIR,
    JUDGE_CACHE_DIR,
]:
    os.makedirs(folder, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("INDEX_DIR    :", INDEX_DIR)

Reading chunks from : /content/drive/MyDrive/RAGBenchmark/chunks
Saving embeddings to: /content/drive/MyDrive/RAGBenchmark/embeddings
PROJECT_ROOT : /content/drive/MyDrive/RAGBenchmark
INDEX_DIR    : /content/drive/MyDrive/RAGBenchmark/indexes


In [13]:
embedding_count = 0

for root, dirs, files in os.walk(EMBEDDING_DIR):
    for f in files:
        if f.endswith(".npz"):
            embedding_count += 1

print("Embeddings:", embedding_count)

Embeddings: 72


In [14]:
#Verify Chunks

chunk_count = 0

for root, dirs, files in os.walk(CHUNK_DIR):
    for f in files:
        if f.endswith(".pkl"):
            chunk_count += 1

print("Chunks:", chunk_count)

Chunks: 36


In [15]:
summary = []

for domain in os.listdir(EMBEDDING_DIR):

    domain_path = os.path.join(
        EMBEDDING_DIR,
        domain
    )

    if not os.path.isdir(domain_path):
        continue

    output_domain = os.path.join(
        INDEX_DIR,
        domain
    )

    os.makedirs(
        output_domain,
        exist_ok=True
    )

    for file in tqdm(
        os.listdir(domain_path),
        desc=domain
    ):

        if not file.endswith(".npz"):
            continue

        embedding_path = os.path.join(
            domain_path,
            file
        )

        index_name = file.replace(
            ".npz",
            ".faiss"
        )

        index_path = os.path.join(
            output_domain,
            index_name
        )

        # Resume-safe
        if os.path.exists(index_path):
            print(
                "Skipping:",
                index_name
            )
            continue

        data = np.load(
            embedding_path,
            allow_pickle=True
        )

        embeddings = data["embeddings"]

        embeddings = embeddings.astype(
            "float32"
        )

        dimension = embeddings.shape[1]

        # Cosine Similarity
        faiss.normalize_L2(
            embeddings
        )

        index = faiss.IndexFlatIP(
            dimension
        )

        index.add(
            embeddings
        )

        faiss.write_index(
            index,
            index_path
        )

        summary.append({
            "domain": domain,
            "index": index_name,
            "vectors": len(
                embeddings
            ),
            "dimension": dimension,
            "path": index_path
        })

        print(
            f"Saved: {index_name}"
        )

biomedical:   0%|          | 0/12 [00:00<?, ?it/s]

Skipping: covidqa_standard_llm_embedder.faiss
Skipping: covidqa_metadata_llm_embedder.faiss
Skipping: covidqa_small2big_llm_embedder.faiss
Skipping: pubmedqa_standard_llm_embedder.faiss
Skipping: pubmedqa_metadata_llm_embedder.faiss
Skipping: pubmedqa_small2big_llm_embedder.faiss
Saved: covidqa_small2big_bge_base.faiss
Saved: covidqa_metadata_bge_base.faiss
Saved: pubmedqa_small2big_bge_base.faiss
Saved: pubmedqa_standard_bge_base.faiss
Saved: covidqa_standard_bge_base.faiss
Saved: pubmedqa_metadata_bge_base.faiss


general:   0%|          | 0/24 [00:00<?, ?it/s]

Skipping: hotpotqa_standard_llm_embedder.faiss
Skipping: hotpotqa_metadata_llm_embedder.faiss
Skipping: hotpotqa_small2big_llm_embedder.faiss
Skipping: msmarco_standard_llm_embedder.faiss
Skipping: msmarco_metadata_llm_embedder.faiss
Skipping: msmarco_small2big_llm_embedder.faiss
Skipping: hagrid_standard_llm_embedder.faiss
Skipping: hagrid_metadata_llm_embedder.faiss
Skipping: hagrid_small2big_llm_embedder.faiss
Skipping: expertqa_standard_llm_embedder.faiss
Skipping: expertqa_metadata_llm_embedder.faiss
Skipping: expertqa_small2big_llm_embedder.faiss
Saved: msmarco_small2big_bge_base.faiss
Saved: expertqa_small2big_bge_base.faiss
Saved: hagrid_small2big_bge_base.faiss
Saved: expertqa_standard_bge_base.faiss
Saved: expertqa_metadata_bge_base.faiss
Saved: hagrid_metadata_bge_base.faiss
Saved: hagrid_standard_bge_base.faiss
Saved: hotpotqa_standard_bge_base.faiss
Saved: msmarco_standard_bge_base.faiss
Saved: msmarco_metadata_bge_base.faiss
Saved: hotpotqa_small2big_bge_base.faiss
Saved:

legal:   0%|          | 0/6 [00:00<?, ?it/s]

Skipping: cuad_standard_llm_embedder.faiss
Skipping: cuad_metadata_llm_embedder.faiss
Skipping: cuad_small2big_llm_embedder.faiss
Saved: cuad_metadata_bge_base.faiss
Saved: cuad_standard_bge_base.faiss
Saved: cuad_small2big_bge_base.faiss


support:   0%|          | 0/18 [00:00<?, ?it/s]

Skipping: delucionqa_standard_llm_embedder.faiss
Skipping: delucionqa_metadata_llm_embedder.faiss
Skipping: delucionqa_small2big_llm_embedder.faiss
Skipping: emanual_standard_llm_embedder.faiss
Skipping: emanual_metadata_llm_embedder.faiss
Skipping: emanual_small2big_llm_embedder.faiss
Skipping: techqa_standard_llm_embedder.faiss
Skipping: techqa_metadata_llm_embedder.faiss
Skipping: techqa_small2big_llm_embedder.faiss
Saved: techqa_small2big_bge_base.faiss
Saved: techqa_metadata_bge_base.faiss
Saved: emanual_standard_bge_base.faiss
Saved: techqa_standard_bge_base.faiss
Saved: delucionqa_small2big_bge_base.faiss
Saved: emanual_small2big_bge_base.faiss
Saved: emanual_metadata_bge_base.faiss
Saved: delucionqa_metadata_bge_base.faiss
Saved: delucionqa_standard_bge_base.faiss


finance:   0%|          | 0/12 [00:00<?, ?it/s]

Skipping: finqa_standard_llm_embedder.faiss
Skipping: finqa_metadata_llm_embedder.faiss
Skipping: finqa_small2big_llm_embedder.faiss
Skipping: tatqa_standard_llm_embedder.faiss
Skipping: tatqa_metadata_llm_embedder.faiss
Skipping: tatqa_small2big_llm_embedder.faiss
Saved: tatqa_standard_bge_base.faiss
Saved: finqa_small2big_bge_base.faiss
Saved: finqa_standard_bge_base.faiss
Saved: finqa_metadata_bge_base.faiss
Saved: tatqa_small2big_bge_base.faiss
Saved: tatqa_metadata_bge_base.faiss


In [16]:
#Summary

import pandas as pd

summary_df = pd.DataFrame(
    summary
)

summary_df.to_csv(
    os.path.join(
        INDEX_DIR,
        "index_summary.csv"
    ),
    index=False
)

summary_df.head()

,domain,index,vectors,dimension,path
0,biomedical,covidqa_small2big_bge_base.faiss,2,768,/content/drive/MyDrive/RAGBenchmark/indexes/bi...
1,biomedical,covidqa_metadata_bge_base.faiss,1,768,/content/drive/MyDrive/RAGBenchmark/indexes/bi...
2,biomedical,pubmedqa_small2big_bge_base.faiss,1,768,/content/drive/MyDrive/RAGBenchmark/indexes/bi...
3,biomedical,pubmedqa_standard_bge_base.faiss,1,768,/content/drive/MyDrive/RAGBenchmark/indexes/bi...
4,biomedical,covidqa_standard_bge_base.faiss,1,768,/content/drive/MyDrive/RAGBenchmark/indexes/bi...


In [17]:
#Verification

index_count = 0

for root, dirs, files in os.walk(INDEX_DIR):
    for f in files:
        if f.endswith(".faiss"):
            index_count += 1

print("Indexes:", index_count)

Indexes: 72


In [18]:
#Smoke Test

sample_index = None

for root, dirs, files in os.walk(INDEX_DIR):

    for file in files:

        if file.endswith(".faiss"):

            sample_index = os.path.join(
                root,
                file
            )

            break

    if sample_index:
        break

print("Testing:", sample_index)

index = faiss.read_index(
    sample_index
)

print(
    "Total vectors:",
    index.ntotal
)

print(
    "Dimension:",
    index.d
)

Testing: /content/drive/MyDrive/RAGBenchmark/indexes/biomedical/covidqa_standard_llm_embedder.faiss
Total vectors: 1
Dimension: 768
